# Locating maximum changes in temperature across resolutions

I noticed that the CSHT dont change much between resolutions, and EKE change the most offshore from the shelf.

At this point, it would be usefull to locate where the tempertures of the model differ the most in the Southern Ocean. Lets try to do that

In [1]:
import cosima_cookbook as cc
import matplotlib.pyplot as plt
import cmocean as cm
import numpy as np
from dask.distributed import Client
from scipy.interpolate import interp1d
from gsw import sigma0, sigma1, sigma2, CT_from_pt, SA_from_SP, p_from_z

import xarray as xr
import cf_xarray as cfxr

In [2]:
client = Client(threads_per_worker = 1)

2025-09-05 09:26:44,344 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:35549' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 9, 72), ('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 8, 40), ('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 9, 17), ('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 9, 81), ('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 8, 49), ('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 8, 58), ('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 8, 3), ('concatenate-getitem-open_dataset-thetao-mean_chunk-e9dff2de1ed20d9e4c06a504249b5c92', 7, 3, 8, 

In [3]:
#panan sessions
session_p01_orig = cc.database.create_session('/home/156/wf4500/databases/access/panan01_original_ol01.db')
session_p005 = cc.database.create_session('/home/156/wf4500/databases/access/panan005_rerun.db') #panan005
session_p0025 = cc.database.create_session('/home/156/wf4500/databases/access/panan0025_final_ol01.db')


start_time = '1999-01-01'
end_time = '2001-01-01'
time_slice=slice(start_time, end_time)


In [4]:
lat_slice  = slice(-83,-59)

figdir = '/g/data/ik11/users/wf4500/Project_panan/GH/Panan_HT_ASC/figs/'

Let's import the temperatures in the model

In [5]:
#lets import the the density to see how it changes along isobaths
#Importing Potential temperature in panan01
T01= cc.querying.getvar('panant-01-zstar-ACCESSyr2','thetao',session_p01_orig,frequency='1 monthly',ncfile = '%ocean_month_z%',\
                            start_time=start_time,end_time=end_time,chunks={}).sel(time = slice( start_time, end_time)).sel(yh=lat_slice).mean('time')
#Importing Potential temperature in panan005
T005= cc.querying.getvar('panant-005-zstar-ACCESSyr2','thetao',session_p005,frequency='1 monthly',ncfile = '%ocean_month_z%',\
                            start_time=start_time,end_time=end_time,chunks={}).sel(time = slice( start_time, end_time)).sel(yh=lat_slice).mean('time')
#Importing Potential temperature in panan0025
T0025= cc.querying.getvar('panant-0025-zstar-ACCESSyr2','thetao',session_p0025,frequency='1 monthly',ncfile = '%ocean_month_z%',\
                            start_time=start_time,end_time=end_time,chunks={}).sel(time = slice( start_time, end_time)).sel(yh=lat_slice).mean('time')

In [ ]:
%%time
T0025.load()
T005.load()
T01.load()
print('all data loaded')

2025-09-05 09:26:44,248 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.07/lib/python3.10/site-packages/distributed/comm/tcp.py", line 225, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.07/lib/python3.10/site-packages/distributed/worker.py", line 1269, in heartbeat
    response = await retry_operation(
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.07/lib/python3.10/site-packages/distributed/utils_comm.py", line 441, in retry_operation
    return await retry(
  File "/g/data/hh5/public/apps/miniconda3/envs/analysis3-24.07/lib/python3.10/site-packages/distributed/utils_comm.py", line 420, in retry


Interpolating into the 1/40th grid

In [ ]:
%%time
T01_into0025 = T01.interp(xh=T0025.xh, yh=T0025.yh)
T005_into0025 = T005.interp(xh=T0025.xh, yh=T0025.yh)

Calculating differences

In [ ]:
T0025minusT01 = T0025-T01_into0025
T005minusT01 = T005_into0025-T01_into0025

Maximum vertical differences in temperatures and their depth

In [ ]:
#extracting first the argmax of the absolute values, to catach maximums in both cooling and warming
T005minusT01_argmax = ((T005minusT01**2)**0.5).fillna(0).argmax('z_l')
T005minusT01_depthofmaxdeltaT = T005minusT01.z_l.isel(z_l=T005minusT01_argmax)
T005minusT01_maxdeltaT = T005minusT01.isel(z_l=T005minusT01_argmax)
landmask = (0*T005minusT01_maxdeltaT+1)
T005minusT01_depthofmaxdeltaT = T005minusT01_depthofmaxdeltaT* landmask

In [ ]:
T005minusT01_depthofmaxdeltaT.plot(vmin=00,vmax=500,cmap=cm.cm.deep)

In [ ]:
plt.figure(figsize=(19,12))
plt.subplots_adjust(left=0.1, right=0.9, 
                    top=0.9, bottom=0.1, 
                    wspace=0.1, hspace=0.3)


ax321 = plt.subplot(3,2,1)
T005minusT01_maxdeltaT.plot()
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('[a] Maximum temperature difference')
ax321.set_facecolor('lightgray')

ax322 = plt.subplot(3,2,2)
T005minusT01_depthofmaxdeltaT.plot(vmin=0,vmax=1000,cmap=cm.cm.deep)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('[b] Depth of maximum temperature difference')
ax322.set_facecolor('lightgray')